In [ ]:
# This is a Jupyter notebook. Create a new Jupyter notebook and paste this content.

"""
# Results Analysis and Visualization

This notebook analyzes experiment results and creates visualizations.
"""

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import scipy.stats as stats
from scipy.stats import f_oneway, chi2_contingency
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# 1. Load Results
print("Loading experiment results...")

# Find latest results directory
results_dir = Path('../data/results')
experiment_dirs = sorted([d for d in results_dir.iterdir() if d.is_dir() and d.name.startswith('experiment_')])

if not experiment_dirs:
    print("No experiment results found!")
    print("Please run an experiment first using: python run_experiment.py")
    raise FileNotFoundError("No experiment results found")

latest_experiment = experiment_dirs[-1]
print(f"Loading results from: {latest_experiment}")

# Load results
results_file = latest_experiment / 'final_results.json'
with open(results_file, 'r') as f:
    results = json.load(f)

# Load summary
summary_file = latest_experiment / 'summary_report.json'
with open(summary_file, 'r') as f:
    summary = json.load(f)

print(f"Loaded {len(results)} experiment results")
print(f"Experiment timestamp: {latest_experiment.name}")

# 2. Convert to DataFrame
print("\n=== Converting to DataFrame ===")

from src.utils.data_loader import results_to_dataframe

df = results_to_dataframe(results)

print(f"DataFrame shape: {df.shape}")
print("\nDataFrame columns:")
print(df.columns.tolist())

print("\nFirst few rows:")
print(df.head())

# 3. Basic Statistics
print("\n=== Basic Statistics ===")

print(f"Total experiments: {len(df)}")
print(f"Number of unique models: {df['model'].nunique()}")
print(f"Number of defense strategies: {df['defense_strategy'].nunique()}")
print(f"Number of manipulation types: {df['manipulation_type'].nunique()}")

# Safety statistics
safe_count = df['is_safe'].sum()
unsafe_count = len(df) - safe_count
safety_rate = safe_count / len(df)

print(f"\nSafety Statistics:")
print(f"Safe responses: {safe_count} ({safety_rate:.1%})")
print(f"Unsafe responses: {unsafe_count} ({1-safety_rate:.1%})")

# 4. Attack Success Rate Analysis
print("\n=== Attack Success Rate Analysis ===")

# ASR by model
asr_by_model = df.groupby('model').apply(
    lambda x: 1 - (x['is_safe'].sum() / len(x))
).sort_values()

print("Attack Success Rate by Model:")
for model, asr in asr_by_model.items():
    print(f"  {model}: {asr:.2%}")

# ASR by defense strategy
asr_by_defense = df.groupby('defense_strategy').apply(
    lambda x: 1 - (x['is_safe'].sum() / len(x))
).sort_values()

print("\nAttack Success Rate by Defense Strategy:")
for defense, asr in asr_by_defense.items():
    print(f"  {defense}: {asr:.2%}")

# ASR by manipulation type
asr_by_manipulation = df.groupby('manipulation_type').apply(
    lambda x: 1 - (x['is_safe'].sum() / len(x))
).sort_values(ascending=False)

print("\nAttack Success Rate by Manipulation Type:")
for manip_type, asr in asr_by_manipulation.items():
    print(f"  {manip_type}: {asr:.2%}")

# 5. Statistical Tests
print("\n=== Statistical Analysis ===")

# ANOVA: Defense strategy effectiveness on safety scores
print("ANOVA: Defense Strategy Effectiveness")
groups = [group['safety_score'].values for name, group in df.groupby('defense_strategy')]
f_stat, p_value = f_oneway(*groups)
print(f"  F-statistic: {f_stat:.4f}")
print(f"  p-value: {p_value:.4e}")
print(f"  Significant difference between defense strategies: {p_value < 0.05}")

# Chi-square: Association between defense strategy and safety
print("\nChi-square: Defense Strategy × Safety")
contingency = pd.crosstab(df['defense_strategy'], df['is_safe'])
chi2, chi_p, dof, expected = chi2_contingency(contingency)
print(f"  Chi2: {chi2:.4f}")
print(f"  p-value: {chi_p:.4e}")
print(f"  Degrees of freedom: {dof}")
print(f"  Significant association: {chi_p < 0.05}")

# 6. Visualization
print("\n=== Creating Visualizations ===")

fig = plt.figure(figsize=(20, 16))

# Plot 1: ASR by Model
ax1 = plt.subplot(3, 3, 1)
asr_by_model.plot(kind='bar', ax=ax1, color='coral')
ax1.set_title('Attack Success Rate by Model', fontsize=14, fontweight='bold')
ax1.set_ylabel('Attack Success Rate', fontsize=12)
ax1.set_xlabel('Model', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

# Add value labels
for i, v in enumerate(asr_by_model.values):
    ax1.text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=10)

# Plot 2: ASR by Defense Strategy
ax2 = plt.subplot(3, 3, 2)
asr_by_defense.plot(kind='bar', ax=ax2, color='skyblue')
ax2.set_title('Attack Success Rate by Defense Strategy', fontsize=14, fontweight='bold')
ax2.set_ylabel('Attack Success Rate', fontsize=12)
ax2.set_xlabel('Defense Strategy', fontsize=12)
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3)

for i, v in enumerate(asr_by_defense.values):
    ax2.text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=10)

# Plot 3: ASR by Manipulation Type
ax3 = plt.subplot(3, 3, 3)
asr_by_manipulation.plot(kind='bar', ax=ax3, color='lightgreen')
ax3.set_title('Attack Success Rate by Manipulation Type', fontsize=14, fontweight='bold')
ax3.set_ylabel('Attack Success Rate', fontsize=12)
ax3.set_xlabel('Manipulation Type', fontsize=12)
ax3.tick_params(axis='x', rotation=45)
ax3.grid(True, alpha=0.3)

for i, v in enumerate(asr_by_manipulation.values):
    ax3.text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=10)

# Plot 4: Safety Score Distribution
ax4 = plt.subplot(3, 3, 4)
df['safety_score'].hist(bins=30, ax=ax4, edgecolor='black', alpha=0.7)
ax4.set_title('Distribution of Safety Scores', fontsize=14, fontweight='bold')
ax4.set_xlabel('Safety Score', fontsize=12)
ax4.set_ylabel('Frequency', fontsize=12)
ax4.grid(True, alpha=0.3)
ax4.axvline(df['safety_score'].mean(), color='red', linestyle='--', 
           label=f'Mean: {df["safety_score"].mean():.2f}')
ax4.legend()

# Plot 5: Response Time by Model
ax5 = plt.subplot(3, 3, 5)
df.boxplot(column='response_time', by='model', ax=ax5)
ax5.set_title('Response Time by Model', fontsize=14, fontweight='bold')
ax5.set_xlabel('Model', fontsize=12)
ax5.set_ylabel('Response Time (seconds)', fontsize=12)
ax5.tick_params(axis='x', rotation=45)
ax5.grid(True, alpha=0.3)

# Plot 6: Safety Score by Defense Strategy
ax6 = plt.subplot(3, 3, 6)
df.boxplot(column='safety_score', by='defense_strategy', ax=ax6)
ax6.set_title('Safety Score by Defense Strategy', fontsize=14, fontweight='bold')
ax6.set_xlabel('Defense Strategy', fontsize=12)
ax6.set_ylabel('Safety Score', fontsize=12)
ax6.tick_params(axis='x', rotation=45)
ax6.grid(True, alpha=0.3)

# Plot 7: Heatmap - Model × Defense Strategy
ax7 = plt.subplot(3, 3, 7)
heatmap_data = df.pivot_table(
    values='safety_score',
    index='model',
    columns='defense_strategy',
    aggfunc='mean'
)
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax7)
ax7.set_title('Average Safety Score: Model × Defense Strategy', fontsize=14, fontweight='bold')
ax7.set_xlabel('Defense Strategy', fontsize=12)
ax7.set_ylabel('Model', fontsize=12)

# Plot 8: Heatmap - Manipulation Type × Defense Strategy
ax8 = plt.subplot(3, 3, 8)
heatmap_data2 = df.pivot_table(
    values='is_safe',
    index='manipulation_type',
    columns='defense_strategy',
    aggfunc='mean'
) * 100  # Convert to percentage
sns.heatmap(heatmap_data2, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=ax8)
ax8.set_title('Safety Rate (%): Manipulation × Defense Strategy', fontsize=14, fontweight='bold')
ax8.set_xlabel('Defense Strategy', fontsize=12)
ax8.set_ylabel('Manipulation Type', fontsize=12)

# Plot 9: Quality vs Safety Trade-off
ax9 = plt.subplot(3, 3, 9)
scatter = ax9.scatter(df['safety_score'], df['coherence_score'], 
                     c=df['response_time'], alpha=0.6, cmap='viridis')
ax9.set_title('Safety vs Quality Trade-off', fontsize=14, fontweight='bold')
ax9.set_xlabel('Safety Score', fontsize=12)
ax9.set_ylabel('Coherence Score (Quality)', fontsize=12)
ax9.grid(True, alpha=0.3)

# Add colorbar for response time
plt.colorbar(scatter, ax=ax9, label='Response Time (seconds)')

plt.tight_layout()
plt.savefig(latest_experiment / 'comprehensive_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# 7. Detailed Analysis
print("\n=== Detailed Analysis ===")

# Most successful attack patterns
print("\nMost Successful Attack Patterns:")
successful_attacks = df[~df['is_safe']]
if len(successful_attacks) > 0:
    successful_patterns = successful_attacks.groupby(
        ['manipulation_type', 'manipulation_level', 'defense_strategy']
    ).size().reset_index(name='count').sort_values('count', ascending=False)
    
    print(successful_patterns.head(10))
else:
    print("No successful attacks found!")

# Most effective defense strategies
print("\nMost Effective Defense Strategies:")
effective_defenses = df.groupby('defense_strategy').agg({
    'is_safe': 'mean',
    'safety_score': 'mean',
    'coherence_score': 'mean'
}).round(3).sort_values('is_safe', ascending=False)

print(effective_defenses)

# 8. Save Analysis Results
print("\n=== Saving Analysis Results ===")

analysis_results = {
    'summary_statistics': {
        'total_experiments': len(df),
        'safe_responses': int(safe_count),
        'unsafe_responses': int(unsafe_count),
        'overall_safety_rate': float(safety_rate),
        'overall_attack_success_rate': float(1 - safety_rate)
    },
    'asr_by_model': asr_by_model.to_dict(),
    'asr_by_defense': asr_by_defense.to_dict(),
    'asr_by_manipulation': asr_by_manipulation.to_dict(),
    'statistical_tests': {
        'anova_defense_strategy': {
            'f_statistic': float(f_stat),
            'p_value': float(p_value),
            'significant': p_value < 0.05
        },
        'chi_square_defense_safety': {
            'chi2': float(chi2),
            'p_value': float(chi_p),
            'dof': int(dof),
            'significant': chi_p < 0.05
        }
    },
    'effective_defenses': effective_defenses.to_dict('index'),
    'successful_attack_patterns': successful_patterns.head(10).to_dict('records') if len(successful_attacks) > 0 else []
}

# Save analysis
analysis_file = latest_experiment / 'detailed_analysis_results.json'
with open(analysis_file, 'w') as f:
    json.dump(analysis_results, f, indent=2)

print(f"Analysis results saved to {analysis_file}")

# 9. Generate Report
print("\n=== Analysis Report ===")
print(f"\nExperiment: {latest_experiment.name}")
print(f"Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nKey Findings:")
print(f"1. Overall Attack Success Rate: {(1 - safety_rate):.1%}")
print(f"2. Most vulnerable model: {asr_by_model.idxmax()} (ASR: {asr_by_model.max():.1%})")
print(f"3. Most effective defense: {asr_by_defense.idxmin()} (ASR: {asr_by_defense.min():.1%})")
print(f"4. Most successful manipulation: {asr_by_manipulation.idxmax()} (ASR: {asr_by_manipulation.max():.1%})")
print(f"5. Statistical significance (defense strategies): {'YES' if p_value < 0.05 else 'NO'}")

print("\nRecommendations:")
print("1. Implement the most effective defense strategies identified")
print("2. Focus on defending against the most successful manipulation types")
print("3. Consider the trade-off between safety and response quality")
print("4. Regularly update defenses as new attack patterns emerge")

print("\nAnalysis complete!")

# 03 Results Analysis

This notebook analyzes the results of the experiments.

## Load results

```python
import json

with open('../data/results/results.json', 'r') as f:
    results = json.load(f)

with open('../data/results/evaluation.json', 'r') as f:
    evaluation = json.load(f)